Gets the packages installed that we need for this code to work

In [ ]:
!pip install -q numpy pandas

This block of code will ask you to upload a file, then it will clean, filter, and normalize the data for you. It will also put it in a format ready for MetaboAnalyst's Biomarker Analysis.

In [ ]:
import sys
import os
import traceback
import numpy as np
import pandas as pd
from google.colab import files

# Asks you to upload the file and stores it at as Dataframe to work with
upfile = files.upload()
upfilename = list(userinfile.keys())[0]
infile = pd.read_excel(upfilename, skiprows=4)

# Drops all unknown features from the dataset
notunknown = infile[infile["Metabolite name"] != "Unknown"]

# Cleans up the metabolite names so that they are parseable by MetaboAnalyst
notunknown["Metabolite name"] = notunknown["Metabolite name"].str.removeprefix("low score: ")
notunknown["Metabolite name"] = notunknown["Metabolite name"].str.removeprefix("no MS2: ")

# Calculates the CV value and then drops all features that exceed the threshold
notunknown["CV"] = notunknown["QC.1"] / notunknown["QC"]
nobadcv = notunknown[notunknown["CV"] <= 0.3]

# Takes away all columns that aren't the sample concentrations
noxtracols = nobadcv.drop(columns=["Alignment ID", "Average Rt(min)", "Adduct type", "Post curation result", "Fill %", "MS/MS assigned", "Reference RT", "Reference m/z", "Formula", "Ontology", "INCHIKEY", "SMILES", "Annotation tag (VS1.0)", "RT matched", "m/z matched", "MS/MS matched", "Comment", "Manually modified for quantification", "Manually modified for annotation", "Isotope tracking parent ID", "Isotope tracking weight number", "RT similarity", "m/z similarity", "Simple dot product", "Weighted dot product", "Reverse dot product", "Matched peaks count", "Matched peaks percentage", "Total score", "S/N average", "Spectrum reference file name", "MS1 isotopic spectrum", "MS/MS spectrum"])

# This figures out how many samples there are, and then gets that slice of the dataset so that we can normalize it.
colnames = noxtracols.columns.to_list()
startcol = 0
stopcol = 0

for col in colnames:
    if "Pool-QC_4" in col:
        targetcol = colnames.index(col) + 1

    elif col == "Blank":
        stopcol = colnames.index(col)

# This subtracts the blanks' average from the sample concentration.
blanknorm = noxtracols.iloc[:, targetcol:stopcol].sub(noxtracols["Blank"], axis=0)

# This handles all zero and negative values by assigning them NaN, 
# then drops all features that have too many missing values or didn't appear in over 50% of samples
maskednorm = blanknorm.mask(blanknorm <= 0)
filteredmask = maskednorm.dropna(axis=0, thresh=len(maskednorm.columns)*.5)

# This applies a half-minimum value to all of the remaining missing values so they normalize correctly
hmv = filteredmask.min(axis=1) / 2 
minvalmask = filteredmask.T.fillna(hmv).T

# Normalizes the data by Log2
lognorm = np.log2(minvalmask)

# This re-attaches the Metabolite name to the corresponding row of the feature
lognorm.insert(0, "Metabolite name", noxtracols.loc[lognorm.index, "Metabolite name"].values)
#lognorm.insert(1, "m.z", noxtracols.loc[lognorm.index, "Average Mz"].values) 

# This gets rid of duplicates; putting this at the end ensures no false negatives from the first instance of the feature being an outlier.
nodupes = lognorm.drop_duplicates(subset=["Metabolite name"])

# This scrapes the Sample Group from the end of the sample name after the last underscore, 
# ie if your sample is named 345_L_V, it would be assigned the V group. It then adds it as a new row under the column headers.
groupslist =[" "]
nameslist = nodupes.columns.to_list()

for i in nameslist[1:]:
    groupslist.append(i.split('_')[-1])

newframe = pd.Series(groupslist, index=nodupes.columns).to_frame().T

finaldf = pd.concat([newframe, nodupes.iloc[0:, :]])

# Outputs a new CSV file ready to be put into the Biomarker Analysis tool
finaldf.to_csv(f"Cleaned_{upfilename}.csv", index=False)

